In [40]:
import os
import json
import urllib.request
import urllib.parse
import requests
import pandas as pd
from requests.auth import HTTPBasicAuth
from pybliometrics.scopus import ScopusSearch
from pybliometrics.utils import create_config, init

# ==========================================
# 1. SETUP LOCAL DIRECTORY
# ==========================================
# This creates the 'MuTech/01_Raw_Exports' folder wherever you run the script
current_dir = os.getcwd()
folder_path = os.path.join(current_dir, 'MuTech', '01_Raw_Exports')
os.makedirs(folder_path, exist_ok=True)
print(f"Ready to save files locally to: {folder_path}\n")

# ==========================================
# 2. SCOPUS EXTRACTION & AUTOMATED FILTERING
# ==========================================
print("\nStarting Scopus extraction...")
try:
    init() 
except FileNotFoundError:
    print("Scopus config not found. Please run create_config() in your terminal manually.")

try:
    scopus_query = 'TITLE-ABS-KEY(("music" OR "instrument" OR "music information retrieval" OR "audio signal processing") AND ("performance" OR "solo" OR "ensemble" OR "transcription") AND ("machine learning" OR "AI" OR "artificial intelligence" OR "deep learning")) AND PUBYEAR > 2019'
    search_results = ScopusSearch(scopus_query)

    if search_results.results:
        df_scopus = pd.DataFrame(search_results.results)
        initial_count = len(df_scopus)
        
        # Automated Filtering: Keep only empirical studies (drops books/reviews)
        valid_types = ['Article', 'Conference Paper']
        df_scopus = df_scopus[df_scopus['subtypeDescription'].isin(valid_types)]
        
        # Automated Filtering: Drop rows without an abstract or authors
        df_scopus = df_scopus.dropna(subset=['description', 'creator'])
        
        # Pruning and Renaming Columns for Rayyan
        df_scopus = df_scopus[['title', 'description', 'coverDate', 'doi', 'creator', 'authkeywords', 'subtypeDescription']]
        df_scopus = df_scopus.rename(columns={
            'title': 'Title',
            'description': 'Abstract',
            'coverDate': 'Year',
            'doi': 'DOI',
            'creator': 'Authors',
            'authkeywords': 'Keywords',
            'subtypeDescription': 'Document Type'
        })
        
        # Clean the Year format
        df_scopus['Year'] = df_scopus['Year'].astype(str).str[:4]
        df_scopus['Source'] = 'Scopus'

        final_count = len(df_scopus)
        print(f"Total Scopus papers originally extracted: {initial_count}")
        print(f"Total Scopus papers kept after automated exclusion: {final_count}")
        print(f"Junk papers automatically dropped: {initial_count - final_count}")
        
        if not df_scopus.empty:
            df_scopus.to_csv(os.path.join(folder_path, 'Scopus_Filtered.csv'), index=False)
            print("Scopus filtered CSV saved successfully.")
    else:
        print("No Scopus results found or query failed.")
except Exception as e:
    print(f"Scopus extraction failed: {e}")


print("\nAll extractions complete! Check the MuTech/01_Raw_Exports folder.")

Ready to save files locally to: c:\Users\mnari\OneDrive\Desktop\MuTech\MuTech\01_Raw_Exports


Starting Scopus extraction...
Scopus extraction failed: Invalid API Key

All extractions complete! Check the MuTech/01_Raw_Exports folder.


In [41]:
import urllib.request
import urllib.parse
import json
import pandas as pd

# ==========================================
# 5. IEEE XPLORE EXTRACTION (UPDATED)
# ==========================================
print("\nStarting IEEE extraction...")
ieee_api_key = "h24uddpgdmamhmxhmgsfhzg3"

try:
    ieee_query = '("music" OR "instrument" OR "music information retrieval" OR "audio signal processing") AND ("performance" OR "solo" OR "ensemble" OR "transcription") AND ("machine learning" OR "AI" OR "artificial intelligence" OR "deep learning")'
    encoded_query = urllib.parse.quote(ieee_query)
    
    # We can use HTTPS here as well for better security
    url = f"https://ieeexploreapi.ieee.org/api/v1/search/articles?apikey={ieee_api_key}&format=json&max_records=200&querytext={encoded_query}"

    # FIX: Add Browser Headers to bypass the 403 Firewall block
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'application/json'
    }

    # Swap urllib for requests to easily pass headers
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        data = response.json()
        print(f"Total IEEE papers extracted: {data.get('total_records', 0)}")
        
        if 'articles' in data:
            df_ieee = pd.DataFrame(data['articles'])
            df_ieee.to_csv(os.path.join(folder_path, 'IEEE_Raw.csv'), index=False)
            print("IEEE CSV saved successfully.")
        else:
            print("No articles found in the IEEE response.")
    else:
        print(f"IEEE API blocked the request. Status Code: {response.status_code}")
        print(f"Server Response: {response.text}")

except Exception as e:
    print(f"IEEE extraction failed: {e}")


Starting IEEE extraction...
IEEE API blocked the request. Status Code: 403
Server Response: <h1>Developer Inactive</h1>


Dataset Consolidation

In [42]:


import pandas as pd

# ==========================================================
# LOAD FILES
# ==========================================================

scopus = pd.read_csv(".\\MuTech\\01_Raw_Exports\\Scopus_Filtered.csv")
ieee = pd.read_csv(".\\MuTech\\01_Raw_Exports\\IEEE_export2026.06.08-04.08.55.csv")

# ==========================================================
# ADD SOURCE DATABASE LABEL
# ==========================================================

scopus["Source Database"] = "Scopus"
ieee["Source Database"] = "IEEE"

# ==========================================================
# STANDARDIZE SCOPUS
# ==========================================================

scopus_clean = pd.DataFrame()

scopus_clean["Title"] = scopus["Title"]
scopus_clean["Abstract"] = scopus["Abstract"]
scopus_clean["Authors"] = scopus["Authors"]
scopus_clean["Year"] = scopus["Year"]
scopus_clean["DOI"] = scopus["DOI"]
scopus_clean["Keywords"] = scopus["Keywords"]

# Scopus uses "Source" for the journal/conference name
scopus_clean["Publication Venue"] = scopus["Source"]

scopus_clean["Document Type"] = scopus["Document Type"]

scopus_clean["Source Database"] = "Scopus"

scopus_clean["PDF Link"] = None

# ==========================================================
# STANDARDIZE IEEE
# ==========================================================

ieee_clean = pd.DataFrame()

ieee_clean["Title"] = ieee["Document Title"]
ieee_clean["Abstract"] = ieee["Abstract"]
ieee_clean["Authors"] = ieee["Authors"]
ieee_clean["Year"] = ieee["Publication Year"]
ieee_clean["DOI"] = ieee["DOI"]

# Merge IEEE keyword fields
ieee_clean["Keywords"] = (
    ieee[
        [
            c
            for c in [
                "Author Keywords",
                "IEEE Terms",
                "Mesh_Terms"
            ]
            if c in ieee.columns
        ]
    ]
    .fillna("")
    .astype(str)
    .agg("; ".join, axis=1)
)

ieee_clean["Publication Venue"] = ieee["Publication Title"]

# Better than using Document Identifier
ieee_clean["Document Type"] = "IEEE Publication"

ieee_clean["Source Database"] = "IEEE"

ieee_clean["PDF Link"] = ieee["PDF Link"]

# ==========================================================
# MERGE DATASETS
# ==========================================================

merged = pd.concat(
    [scopus_clean, ieee_clean],
    ignore_index=True
)

# ==========================================================
# CREATE UNIQUE RECORD ID
# ==========================================================

merged["record_id"] = range(
    1,
    len(merged) + 1
)

# ==========================================================
# CLEAN TEXT FIELDS
# ==========================================================

for col in ["Title", "Abstract", "Authors", "Keywords"]:
    merged[col] = (
        merged[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# ==========================================================
# CREATE NORMALIZED TITLE
# FOR FUTURE DEDUPLICATION
# ==========================================================

merged["Title_norm"] = (
    merged["Title"]
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
)

# ==========================================================
# SAVE MERGED FILE
# ==========================================================

merged.to_csv(
    "merged_scopus_ieee.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Total records: {len(merged)}")
print("Saved: merged_scopus_ieee.csv")

Total records: 9943
Saved: merged_scopus_ieee.csv


In [43]:
# Initial count
initial_count = len(merged)

# ==========================================
# DOI DEDUPLICATION
# ==========================================

after_doi = merged.drop_duplicates(
    subset=["DOI"],
    keep="first"
)

doi_duplicates_removed = initial_count - len(after_doi)

# ==========================================
# TITLE DEDUPLICATION
# ==========================================

after_title = after_doi.drop_duplicates(
    subset=["Title_norm"],
    keep="first"
)

title_duplicates_removed = len(after_doi) - len(after_title)

# ==========================================
# FINAL COUNTS
# ==========================================

total_duplicates_removed = (
    doi_duplicates_removed +
    title_duplicates_removed
)

print(f"Initial records: {initial_count}")
print(f"After DOI deduplication: {len(after_doi)}")
print(f"DOI duplicates removed: {doi_duplicates_removed}")

print(f"\nAfter Title deduplication: {len(after_title)}")
print(f"Title duplicates removed: {title_duplicates_removed}")

print(f"\nTotal duplicates removed: {total_duplicates_removed}")

# Replace merged with final deduplicated dataset
merged = after_title

# ==========================================================
# SAVE DEDUPLICATED FILE
# ==========================================================

merged.to_csv(
    "merged_deduplicated.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Total records: {len(merged)}")
print("Saved: merged_deduplicated.csv")

Initial records: 9943
After DOI deduplication: 9017
DOI duplicates removed: 926

After Title deduplication: 9007
Title duplicates removed: 10

Total duplicates removed: 936
Total records: 9007
Saved: merged_deduplicated.csv


In [44]:
# import pandas as pd

# # Create decision columns
# merged["Auto_Screen"] = "Include"
# merged["Auto_Reason"] = ""

# # ==================================================
# # YEAR FILTER
# # ==================================================

# mask = (merged["Year"] < 2020) | (merged["Year"] > 2026)

# merged.loc[mask, "Auto_Screen"] = "Exclude"
# merged.loc[mask, "Auto_Reason"] = "Outside year range"

# print(f"Excluded based on year: {mask.sum()}")

# # ==================================================
# # ABSTRACT REQUIRED
# # ==================================================

# mask = (
#     merged["Abstract"]
#     .fillna("")
#     .str.strip()
#     .eq("")
# )

# merged.loc[mask, "Auto_Screen"] = "Exclude"
# merged.loc[mask, "Auto_Reason"] = "Missing abstract"

# print(f"Excluded due to missing abstract: {mask.sum()}")

# # ==================================================
# # DOI REQUIRED
# # ==================================================

# mask = (
#     merged["DOI"]
#     .fillna("")
#     .str.strip()
#     .eq("")
# )

# merged.loc[mask, "Auto_Screen"] = "Exclude"
# merged.loc[mask, "Auto_Reason"] = "Missing DOI"

# print(f"Excluded due to missing DOI: {mask.sum()}")

# # ==================================================
# # FULL TEXT AVAILABLE
# # ==================================================

# if "PDF Link" in merged.columns:

#     mask = (
#         merged["PDF Link"]
#         .fillna("")
#         .str.strip()
#         .eq("")
#     )

#     merged.loc[mask, "Auto_Screen"] = "Exclude"
#     merged.loc[mask, "Auto_Reason"] = "No full text"
    
#     print(f"Excluded due to no full text: {mask.sum()}")

# # ==================================================
# # DOCUMENT TYPE FILTER
# # ==================================================

# exclude_types = [
#     "Editorial",
#     "Book Chapter",
#     "Patent",
#     "Standard",
#     "Erratum",
#     "Letter"
# ]

# mask = merged["Document Type"].isin(exclude_types)

# merged.loc[mask, "Auto_Screen"] = "Exclude"
# merged.loc[mask, "Auto_Reason"] = "Wrong publication type"

# print(f"Excluded due to wrong publication type: {mask.sum()}")

# # ==================================================
# # RESULTS
# # ==================================================

# included = merged[
#     merged["Auto_Screen"] == "Include"
# ]

# excluded = merged[
#     merged["Auto_Screen"] == "Exclude"
# ]

# print("Included:", len(included))
# print("Excluded:", len(excluded))

In [45]:
merged["search_text"] = (
    merged["Title"].fillna("") + " " +
    merged["Abstract"].fillna("") + " " +
    merged["Keywords"].fillna("")
).str.lower()

exclude_terms = [
    # Speech
    "speech recognition",
    "speaker recognition",
    "speaker verification",
    "speaker identification",
    "speech synthesis",
    "text-to-speech",
    "tts",
    "automatic speech recognition",
    "voice conversion",
    "speech enhancement",
    "speech separation",
    "spoken language",
    "dialogue system",

    # Biomedical
    "parkinson",
    "alzheimer",
    "dysphonia",
    "pathology",
    "clinical",
    "medical",
    "healthcare",
    "diagnosis",
    "disease",
    "patient",
    "biomarker",
    "hospital",

    # Recommendation
    "recommender system",
    "playlist generation",
    "playlist continuation",
    "user preference",
    "collaborative filtering",
    "music discovery",
    "consumer behavior",
    "listening preference",

    # Industry / Business
    "music marketing",
    "licensing",
    "streaming platform",
    "music consumption",
    "music popularity",
    "chart prediction",
    "social media",
    "music industry",

    # Education / Psychology
    "emotion perception",
    "music cognition",
    "listener study",
    "perceptual study",
    "survey study",
    "questionnaire",

    # Musicology
    "historical music analysis",
    "music evolution",
    "style evolution",
    "musicology",

    # Metadata-focused
    "metadata tagging",
    "artist tagging"
    
    "semantic tagging",
    "genre tagging"
    
    "hearing aid",
    "cochlear implant",
    "audiology",
    "hearing loss",
    "hearing impairment"
    
    "music generation",
    "music therapy",
    "mood detection"
]

mask = merged["search_text"].apply(
    lambda x: any(term in x for term in exclude_terms)
)

filtered = merged[~mask]

print("excluded based on exclude terms:", mask.sum())



C:\Users\mnari\AppData\Local\Temp\ipykernel_48424\223032608.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged["search_text"] = (


excluded based on exclude terms: 2829


In [46]:
def find_matching_term(text, terms):
    for term in terms:
        if term in text:
            return term
    return ""

merged["Exclude_Reason"] = merged["search_text"].apply(
    lambda x: find_matching_term(x, exclude_terms)
)

C:\Users\mnari\AppData\Local\Temp\ipykernel_48424\1925527746.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged["Exclude_Reason"] = merged["search_text"].apply(


In [47]:
priority_terms = [
    "music information retrieval",
    "mir",
    "music retrieval",
    "music classification",
    "genre classification",
    "music tagging",
    "music transcription",
    "automatic music transcription",
    "source separation",
    "music source separation",
    "singing voice",
    "singing",
    "choral",
    "choir",
    "ensemble",
    "instrumental",
    "pitch estimation",
    "pitch detection",
    "music similarity",
    "audio tagging",
    "music emotion recognition",
    "score following",
    "optical music recognition",
    "omr"
]

In [48]:
def contains_any(text, terms):
    return any(term in text for term in terms)

merged["Priority_Flag"] = merged["search_text"].apply(
    lambda x: contains_any(x, priority_terms)
)

merged["Exclude_Flag"] = merged["search_text"].apply(
    lambda x: contains_any(x, exclude_terms)
)



C:\Users\mnari\AppData\Local\Temp\ipykernel_48424\2552450553.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged["Priority_Flag"] = merged["search_text"].apply(
C:\Users\mnari\AppData\Local\Temp\ipykernel_48424\2552450553.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merged["Exclude_Flag"] = merged["search_text"].apply(


In [49]:
priority_df = merged[
    merged["Priority_Flag"]
].copy()

In [50]:
excluded_df = merged[
    merged["Exclude_Flag"]
].copy()

In [51]:
remaining_df = merged[
    (~merged["Priority_Flag"]) &
    (~merged["Exclude_Flag"])
].copy()

In [53]:
priority_df.to_csv(
    "priority_records.csv",
    index=False,
    encoding="utf-8-sig"
)


excluded_df.to_csv(
    "excluded_records.csv",
    index=False,
    encoding="utf-8-sig"
)

remaining_df.to_csv(
    "remaining_records.csv",
    index=False,
    encoding="utf-8-sig"
)

In [ ]:
print(f"Total records: {len(merged):,}")
print(f"Priority records: {len(priority_df):,}")
print(f"Excluded records: {len(excluded_df):,}")
print(f"Remaining records: {len(remaining_df):,}")

Total records: 9,007
Priority records: 1,728
Excluded records: 3,084
Remaining records: 4,710


In [ ]:
official = pd.read_csv(
    r".\MuTech\[CITE4D] MuTech Research Papers - OFFICIAL PAPERS.csv"
)

official["Title_norm"] = (
    official["TOPIC"]
    .fillna("")
    .str.lower()
    .str.strip()
    .str.replace(r"[^\w\s]", "", regex=True)
)

duplicates = merged.merge(
    official[
        [
            "TOPIC",
            "AUTHOR",
            "YEAR",
            "APA FORMAT",
            "Title_norm"
        ]
    ],
    on="Title_norm",
    how="inner"
)

duplicates.to_csv(
    "official_paper_matches.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Matches found: {len(duplicates)}")
print("Saved: official_paper_matches.csv")



In [ ]:
new_candidates = merged[
    ~merged["Title_norm"].isin(
        official["Title_norm"]
    )
]

new_candidates.to_csv(
    "new_candidates.csv",
    index=False
)

print(f"New candidates: {len(new_candidates)}")